# English-Bangla mT5 Back-Translation Experiment on Google Colab GPU

This notebook runs a real measured experiment for the project:

**Research question:** Does back-translation improve English-Bangla translation performance in a low-resource setting?

It uses:
- `google/mt5-small`
- Stable fp32 training on T4 GPU, because fp16 can produce NaN losses for mT5/T5
- AI4Bharat Samanantar Bengali (`ai4bharat/samanantar`, config/data shard `bn`)
- A low-resource subset of 5,000 to 20,000 English-Bangla sentence pairs
- A reverse Bangla-to-English mT5-small model to create synthetic English sources
- BLEU evaluation on a held-out 500-pair test set

No result values are prefilled. Run all cells on a Colab GPU runtime to generate real measured outputs.

## Colab Runtime Instructions

1. Open [Google Colab create notebook](https://colab.research.google.com/#create=true).
2. Upload this `.ipynb` file or open it from Google Drive.
3. Select **Runtime -> Change runtime type -> T4 GPU**.
4. Run the cells from top to bottom.

Expected runtime depends on Colab availability and configuration. This v3 report-run default uses 5,000 original training pairs, 500 validation pairs, 500 test pairs, 500 synthetic pairs, 3 epochs for the baseline/improved forward models, and 2 epochs for the reverse model. It intentionally disables fp16 because the previous T4 fp16 run produced NaN losses and empty predictions. If the run is stable, you can raise `TRAIN_SIZE` and `SYNTHETIC_SIZE`; if memory is tight, keep batch size 4 and gradient accumulation 4.

In [ ]:
# Install dependencies. Colab already includes PyTorch, but the NLP stack is pinned here.
!pip -q install -U "transformers>=4.41,<5" "datasets>=2.19,<4" "sentencepiece>=0.2" "sacrebleu>=2.4" "accelerate>=0.30" "protobuf>=4" "pandas" "matplotlib" 

In [ ]:
import gc
import inspect
import json
import math
import os
import random
import re
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
import sacrebleu
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU before training.")

SEED = 42
set_seed(SEED)
random.seed(SEED)

MODEL_NAME = "google/mt5-small"
DATASET_NAME = "ai4bharat/samanantar"
DATASET_CONFIG = "bn"
DATASET_PARQUET = "hf://datasets/ai4bharat/samanantar/bn/train-00000-of-00005.parquet"

# Scale controls. T4-friendly default; raise TRAIN_SIZE to 20000 if runtime permits.
TRAIN_SIZE = 5000        # Stable T4 default. Raise to 10000 or 20000 after a successful run.
VAL_SIZE = 500
TEST_SIZE = 500
SYNTHETIC_SIZE = 500     # Stable T4 default. Raise to 1000 or 2000 after a successful run.
CANDIDATE_POOL_SIZE = TRAIN_SIZE + VAL_SIZE + TEST_SIZE + SYNTHETIC_SIZE + 1000

BASELINE_EPOCHS = 3
REVERSE_EPOCHS = 2       # Reverse model is used only to create synthetic sources.
IMPROVED_EPOCHS = 3

BATCH_SIZE = 4           # Stable T4 default for fp32 mT5-small. Try 8 if memory allows.
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
MAX_SOURCE_LENGTH = 64
MAX_TARGET_LENGTH = 64
GENERATION_MAX_LENGTH = 64
GENERATION_BATCH_SIZE = 16
BLEU_TOKENIZER = "flores200"
FP16 = False             # mT5/T5 can become numerically unstable in fp16 on T4; keep fp32 for valid losses.

RESULTS_DIR = Path("/content/mt5_bn_backtranslation_results")
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
EXAMPLES_DIR = RESULTS_DIR / "examples"
SCORES_DIR = RESULTS_DIR / "scores"
for directory in [RESULTS_DIR, TABLES_DIR, FIGURES_DIR, EXAMPLES_DIR, SCORES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

config = {
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "dataset_config": DATASET_CONFIG,
    "train_size": TRAIN_SIZE,
    "validation_size": VAL_SIZE,
    "test_size": TEST_SIZE,
    "synthetic_requested": SYNTHETIC_SIZE,
    "candidate_pool_size": CANDIDATE_POOL_SIZE,
    "baseline_epochs": BASELINE_EPOCHS,
    "reverse_epochs": REVERSE_EPOCHS,
    "improved_epochs": IMPROVED_EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "generation_max_length": GENERATION_MAX_LENGTH,
    "bleu_tokenizer": BLEU_TOKENIZER,
    "fp16": FP16,
    "precision_note": "fp32 used intentionally; previous fp16 T4 run produced NaN losses and empty predictions",
    "run_mode": "v3_report_run",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
}
pd.DataFrame([{"Parameter": k, "Value": v} for k, v in config.items()]).to_csv(
    TABLES_DIR / "training_configuration.csv", index=False
)
print(json.dumps(config, indent=2, ensure_ascii=False))

In [ ]:
def clean_text(value):
    if value is None:
        return ""
    text = str(value).replace("\u200c", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def word_count(text):
    return len(str(text).split())


def get_full_dataset_examples():
    url = (
        "https://datasets-server.huggingface.co/info"
        f"?dataset={DATASET_NAME}&config={DATASET_CONFIG}"
    )
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        return int(response.json()["dataset_info"]["splits"]["train"]["num_examples"])
    except Exception as exc:
        print("Could not read full dataset metadata:", exc)
        return None


def stream_clean_pairs(candidate_pool_size):
    print(f"Streaming {candidate_pool_size:,} candidate rows from Samanantar Bengali...")
    dataset = load_dataset("parquet", data_files=DATASET_PARQUET, split="train", streaming=True)
    seen = set()
    pairs = []
    scanned = empty_removed = duplicates_removed = 0
    for row in dataset:
        scanned += 1
        english = clean_text(row.get("src"))
        bangla = clean_text(row.get("tgt"))
        if not english or not bangla:
            empty_removed += 1
            continue
        key = (english, bangla)
        if key in seen:
            duplicates_removed += 1
            continue
        seen.add(key)
        pairs.append({"english": english, "bangla": bangla})
        if len(pairs) >= candidate_pool_size:
            break
    stats = {
        "candidate_rows_scanned": scanned,
        "empty_rows_removed": empty_removed,
        "duplicate_pairs_removed": duplicates_removed,
        "clean_unique_pairs": len(pairs),
    }
    return pairs, stats


def split_pairs(pairs):
    needed = TRAIN_SIZE + VAL_SIZE + TEST_SIZE + SYNTHETIC_SIZE
    if len(pairs) < needed:
        raise ValueError(f"Need at least {needed:,} clean pairs, got {len(pairs):,}.")
    shuffled = list(pairs)
    random.Random(SEED).shuffle(shuffled)
    train_end = TRAIN_SIZE
    val_end = train_end + VAL_SIZE
    test_end = val_end + TEST_SIZE
    mono_end = test_end + SYNTHETIC_SIZE
    return {
        "train": shuffled[:train_end],
        "validation": shuffled[train_end:val_end],
        "test": shuffled[val_end:test_end],
        "monolingual_bn": shuffled[test_end:mono_end],
    }


full_examples = get_full_dataset_examples()
pairs, preprocessing_stats = stream_clean_pairs(CANDIDATE_POOL_SIZE)
splits = split_pairs(pairs)

dataset_rows = [
    {
        "Split": "Full Samanantar bn train metadata",
        "Sentence pairs": full_examples,
        "Avg English length (words)": "Not computed; full corpus not downloaded",
        "Avg Bangla length (words)": "Not computed; full corpus not downloaded",
    }
]
for split_name, rows in [
    ("Train", splits["train"]),
    ("Validation", splits["validation"]),
    ("Test", splits["test"]),
    ("Bangla monolingual pool", splits["monolingual_bn"]),
]:
    dataset_rows.append(
        {
            "Split": split_name,
            "Sentence pairs": len(rows),
            "Avg English length (words)": round(sum(word_count(r["english"]) for r in rows) / len(rows), 2),
            "Avg Bangla length (words)": round(sum(word_count(r["bangla"]) for r in rows) / len(rows), 2),
        }
    )

dataset_stats = pd.DataFrame(dataset_rows)
dataset_stats.to_csv(TABLES_DIR / "dataset_statistics.csv", index=False)
pd.DataFrame(pairs).to_csv(RESULTS_DIR / "cleaned_candidate_pairs.csv", index=False)
print("Preprocessing:", preprocessing_stats)
display(dataset_stats)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_hf_dataset(rows):
    return Dataset.from_pandas(pd.DataFrame(rows), preserve_index=False)


def preprocess_dataset(rows, direction):
    dataset = make_hf_dataset(rows)

    def preprocess_batch(batch):
        if direction == "en_bn":
            sources = [f"translate English to Bengali: {text}" for text in batch["english"]]
            targets = batch["bangla"]
        elif direction == "bn_en":
            sources = [f"translate Bengali to English: {text}" for text in batch["bangla"]]
            targets = batch["english"]
        else:
            raise ValueError(direction)

        model_inputs = tokenizer(
            sources,
            max_length=MAX_SOURCE_LENGTH,
            truncation=True,
        )
        labels = tokenizer(
            text_target=targets,
            max_length=MAX_TARGET_LENGTH,
            truncation=True,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    return dataset.map(
        preprocess_batch,
        batched=True,
        remove_columns=dataset.column_names,
        desc=f"Tokenizing {direction}",
    )


tokenized = {
    "train_en_bn": preprocess_dataset(splits["train"], "en_bn"),
    "val_en_bn": preprocess_dataset(splits["validation"], "en_bn"),
    "train_bn_en": preprocess_dataset(splits["train"], "bn_en"),
    "val_bn_en": preprocess_dataset(splits["validation"], "bn_en"),
}

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer)
print(tokenized)

In [ ]:
def training_args(output_dir, epochs):
    kwargs = {
        "output_dir": str(output_dir),
        "num_train_epochs": epochs,
        "per_device_train_batch_size": BATCH_SIZE,
        "per_device_eval_batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "adafactor": True,
        "max_grad_norm": 1.0,
        "logging_strategy": "steps",
        "logging_steps": 50,
        "save_strategy": "no",
        "report_to": "none",
        "predict_with_generate": False,
        "fp16": FP16,
        "dataloader_num_workers": 2,
        "remove_unused_columns": True,
    }
    sig = inspect.signature(Seq2SeqTrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "epoch"
    else:
        kwargs["evaluation_strategy"] = "epoch"
    return Seq2SeqTrainingArguments(**kwargs)


def train_seq2seq(run_name, train_dataset, eval_dataset, epochs):
    print(f"\n=== Training {run_name} for {epochs} epoch(s) ===")
    started = time.perf_counter()
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model.config.decoder_start_token_id = tokenizer.pad_token_id
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    args = training_args(RESULTS_DIR / "checkpoints" / run_name, epochs)
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    train_output = trainer.train()
    eval_metrics = trainer.evaluate()
    elapsed = time.perf_counter() - started
    if not math.isfinite(float(train_output.training_loss)) or float(train_output.training_loss) <= 0:
        raise RuntimeError(
            f"Invalid training loss for {run_name}: {train_output.training_loss}. "
            "Stop and lower LR/batch size; do not use this run in the report."
        )
    if not math.isfinite(float(eval_metrics["eval_loss"])) or float(eval_metrics["eval_loss"]) <= 0:
        raise RuntimeError(
            f"Invalid validation loss for {run_name}: {eval_metrics['eval_loss']}. "
            "Stop and lower LR/batch size; do not use this run in the report."
        )
    log_history = pd.DataFrame(trainer.state.log_history)
    log_history["run_name"] = run_name
    log_history.to_csv(SCORES_DIR / f"{run_name}_trainer_log_history.csv", index=False)
    summary = {
        "run_name": run_name,
        "epochs": epochs,
        "train_runtime_seconds": round(elapsed, 2),
        "train_loss": float(train_output.training_loss),
        "validation_loss": float(eval_metrics["eval_loss"]),
    }
    print(summary)
    return model, trainer, log_history, summary


def get_bad_words_ids():
    bad = []
    for index in range(100):
        encoded = tokenizer.encode(f"<extra_id_{index}>", add_special_tokens=False)
        if encoded:
            bad.append(encoded)
    return bad


BAD_WORDS_IDS = get_bad_words_ids()


def generate_texts(model, input_texts, source_language):
    model.eval()
    device = model.device
    predictions = []
    prefix = (
        "translate English to Bengali: "
        if source_language == "english"
        else "translate Bengali to English: "
    )
    started = time.perf_counter()
    with torch.no_grad():
        for start in range(0, len(input_texts), GENERATION_BATCH_SIZE):
            batch_texts = [prefix + text for text in input_texts[start:start + GENERATION_BATCH_SIZE]]
            encoded = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_SOURCE_LENGTH,
            ).to(device)
            generated = model.generate(
                **encoded,
                max_new_tokens=GENERATION_MAX_LENGTH,
                num_beams=4,
                do_sample=False,
                bad_words_ids=BAD_WORDS_IDS,
                no_repeat_ngram_size=3,
            )
            predictions.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    elapsed = time.perf_counter() - started
    return [clean_text(p) for p in predictions], elapsed


def nonempty_rate(predictions):
    return sum(bool(str(p).strip()) for p in predictions) / max(1, len(predictions))


def corpus_bleu(predictions, references):
    score = sacrebleu.corpus_bleu(predictions, [references], tokenize=BLEU_TOKENIZER)
    return {
        "bleu": float(score.score),
        "signature": score.format(signature=True),
        "tokenizer": BLEU_TOKENIZER,
    }


def cleanup_model(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
baseline_model, baseline_trainer, baseline_history, baseline_summary = train_seq2seq(
    "baseline_en_bn",
    tokenized["train_en_bn"],
    tokenized["val_en_bn"],
    BASELINE_EPOCHS,
)

test_sources = [row["english"] for row in splits["test"]]
test_refs = [row["bangla"] for row in splits["test"]]
baseline_predictions, baseline_generation_seconds = generate_texts(
    baseline_model,
    test_sources,
    source_language="english",
)
baseline_nonempty = nonempty_rate(baseline_predictions)
print("Baseline non-empty prediction rate:", baseline_nonempty)
if baseline_nonempty < 0.80:
    raise RuntimeError(f"Baseline non-empty prediction rate is only {baseline_nonempty:.2%}. Do not use this run; training/generation is invalid.")
baseline_bleu = corpus_bleu(baseline_predictions, test_refs)
print("Baseline BLEU:", baseline_bleu)

baseline_examples = pd.DataFrame(
    {
        "Source English": test_sources[:10],
        "Reference Bangla": test_refs[:10],
        "Baseline Prediction": baseline_predictions[:10],
    }
)
baseline_examples.to_csv(EXAMPLES_DIR / "baseline_translation_examples.csv", index=False)
display(baseline_examples)

In [ ]:
reverse_model, reverse_trainer, reverse_history, reverse_summary = train_seq2seq(
    "reverse_bn_en_for_backtranslation",
    tokenized["train_bn_en"],
    tokenized["val_bn_en"],
    REVERSE_EPOCHS,
)

mono_bangla = [row["bangla"] for row in splits["monolingual_bn"]]
synthetic_english, synthetic_generation_seconds = generate_texts(
    reverse_model,
    mono_bangla,
    source_language="bangla",
)

synthetic_pairs = []
for bangla, synthetic_en in zip(mono_bangla, synthetic_english):
    synthetic_en = clean_text(synthetic_en)
    if not synthetic_en or "<extra_id" in synthetic_en:
        continue
    synthetic_pairs.append(
        {
            "english": synthetic_en,
            "bangla": bangla,
            "creation_method": "Bangla sentence translated into synthetic English using the reverse mT5-small model.",
        }
    )

synthetic_df = pd.DataFrame(synthetic_pairs)
synthetic_df.to_csv(EXAMPLES_DIR / "synthetic_backtranslation_pairs.csv", index=False)
print(f"Requested synthetic pairs: {SYNTHETIC_SIZE}; usable synthetic pairs generated: {len(synthetic_pairs)}")
synthetic_nonempty = nonempty_rate(synthetic_english)
print("Reverse-model non-empty generation rate:", synthetic_nonempty)
MIN_REQUIRED_SYNTHETIC = min(500, SYNTHETIC_SIZE)
if len(synthetic_pairs) < MIN_REQUIRED_SYNTHETIC:
    raise RuntimeError(
        f"Only {len(synthetic_pairs)} usable synthetic pairs were generated; "
        f"at least {MIN_REQUIRED_SYNTHETIC} are required for this report run. "
        "Do not continue to improved training; train the reverse model longer or improve generation settings."
    )
display(synthetic_df.head(10))

cleanup_model(reverse_model, reverse_trainer)

improved_train_pairs = splits["train"] + [
    {"english": row["english"], "bangla": row["bangla"]} for row in synthetic_pairs
]
tokenized_improved_train = preprocess_dataset(improved_train_pairs, "en_bn")
print("Improved training pairs:", len(improved_train_pairs))

In [ ]:
improved_model, improved_trainer, improved_history, improved_summary = train_seq2seq(
    "improved_en_bn_original_plus_synthetic",
    tokenized_improved_train,
    tokenized["val_en_bn"],
    IMPROVED_EPOCHS,
)

improved_predictions, improved_generation_seconds = generate_texts(
    improved_model,
    test_sources,
    source_language="english",
)
improved_nonempty = nonempty_rate(improved_predictions)
print("Improved non-empty prediction rate:", improved_nonempty)
if improved_nonempty < 0.80:
    raise RuntimeError(f"Improved non-empty prediction rate is only {improved_nonempty:.2%}. Do not use this run; training/generation is invalid.")
improved_bleu = corpus_bleu(improved_predictions, test_refs)
print("Improved BLEU:", improved_bleu)

comparison_examples = pd.DataFrame(
    {
        "Source English": test_sources[:10],
        "Reference Bangla": test_refs[:10],
        "Baseline Prediction": baseline_predictions[:10],
        "Improved Prediction": improved_predictions[:10],
    }
)
comparison_examples.to_csv(TABLES_DIR / "sample_translation_comparison.csv", index=False)
comparison_examples.to_csv(EXAMPLES_DIR / "sample_translation_comparison.csv", index=False)
display(comparison_examples)

In [ ]:
bleu_rows = pd.DataFrame(
    [
        {
            "Model": "Baseline mT5-small",
            "Training data": "Original Samanantar subset",
            "BLEU": round(baseline_bleu["bleu"], 4),
            "BLEU tokenizer": baseline_bleu["tokenizer"],
        },
        {
            "Model": "Improved mT5-small",
            "Training data": "Original subset + synthetic back-translated pairs",
            "BLEU": round(improved_bleu["bleu"], 4),
            "BLEU tokenizer": improved_bleu["tokenizer"],
        },
    ]
)
bleu_rows.to_csv(TABLES_DIR / "bleu_score_comparison.csv", index=False)
bleu_rows.to_csv(SCORES_DIR / "bleu_scores.csv", index=False)

validation_rows = pd.DataFrame(
    [
        {
            "Model": "Baseline mT5-small",
            "Validation loss": round(baseline_summary["validation_loss"], 4),
            "Training loss": round(baseline_summary["train_loss"], 4),
            "Training time seconds": baseline_summary["train_runtime_seconds"],
            "Test generation time seconds": round(baseline_generation_seconds, 2),
        },
        {
            "Model": "Reverse mT5-small",
            "Validation loss": round(reverse_summary["validation_loss"], 4),
            "Training loss": round(reverse_summary["train_loss"], 4),
            "Training time seconds": reverse_summary["train_runtime_seconds"],
            "Test generation time seconds": round(synthetic_generation_seconds, 2),
        },
        {
            "Model": "Improved mT5-small",
            "Validation loss": round(improved_summary["validation_loss"], 4),
            "Training loss": round(improved_summary["train_loss"], 4),
            "Training time seconds": improved_summary["train_runtime_seconds"],
            "Test generation time seconds": round(improved_generation_seconds, 2),
        },
    ]
)
validation_rows.to_csv(TABLES_DIR / "validation_loss_and_training_time.csv", index=False)

all_history = pd.concat([baseline_history, reverse_history, improved_history], ignore_index=True, sort=False)
all_history.to_csv(SCORES_DIR / "training_log_history_all_models.csv", index=False)

manifest = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "runtime_gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
    "config": config,
    "full_samanantar_bn_train_examples": full_examples,
    "preprocessing": preprocessing_stats,
    "synthetic_pairs_generated": len(synthetic_pairs),
    "baseline_bleu": baseline_bleu,
    "improved_bleu": improved_bleu,
    "baseline_nonempty_prediction_rate": baseline_nonempty,
    "improved_nonempty_prediction_rate": improved_nonempty,
    "baseline_summary": baseline_summary,
    "reverse_summary": reverse_summary,
    "improved_summary": improved_summary,
    "limitation_note": (
        "Results are measured from this Colab runtime only. If Colab assigns a non-T4 GPU, "
        "disconnects, or requires smaller batch/data settings, report the actual configuration above."
    ),
}
(RESULTS_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print("BLEU comparison")
display(bleu_rows)
print("Validation/training comparison")
display(validation_rows)

In [ ]:
# Figures
plt.figure(figsize=(7, 4))
plt.bar(bleu_rows["Model"], bleu_rows["BLEU"], color=["#34699a", "#c4552f"])
plt.ylabel("BLEU")
plt.title("BLEU Score Comparison")
for idx, row in bleu_rows.iterrows():
    plt.text(idx, row["BLEU"], f"{row['BLEU']:.4f}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "bleu_comparison_chart.png", dpi=200)
plt.show()

plt.figure(figsize=(7, 4))
plt.bar(validation_rows["Model"], validation_rows["Validation loss"], color=["#34699a", "#6f7f3f", "#c4552f"])
plt.ylabel("Validation loss")
plt.title("Validation Loss Comparison")
for idx, row in validation_rows.iterrows():
    plt.text(idx, row["Validation loss"], f"{row['Validation loss']:.4f}", ha="center", va="bottom")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "validation_loss_comparison_chart.png", dpi=200)
plt.show()

loss_logs = all_history[all_history["loss"].notna()].copy()
plt.figure(figsize=(9, 5))
for run_name, group in loss_logs.groupby("run_name"):
    plt.plot(group["step"], group["loss"], label=run_name)
plt.xlabel("Optimizer step")
plt.ylabel("Training loss")
plt.title("Training Loss by Model")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "training_loss_chart.png", dpi=200)
plt.show()

dataset_plot = pd.DataFrame(
    [
        {"Split": "Train", "Sentence pairs": TRAIN_SIZE},
        {"Split": "Validation", "Sentence pairs": VAL_SIZE},
        {"Split": "Test", "Sentence pairs": TEST_SIZE},
        {"Split": "Synthetic", "Sentence pairs": len(synthetic_pairs)},
    ]
)
plt.figure(figsize=(7, 4))
plt.bar(dataset_plot["Split"], dataset_plot["Sentence pairs"], color="#3f7f6f")
plt.ylabel("Sentence pairs")
plt.title("Dataset Distribution")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dataset_distribution_chart.png", dpi=200)
plt.show()

In [ ]:
discussion = f'''# Hardware, Results, and Limitations

## Hardware

- Runtime GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected"}
- Model: `{MODEL_NAME}`
- Dataset: `{DATASET_NAME}` Bengali/Samanantar shard
- Original training pairs: {TRAIN_SIZE}
- Validation pairs: {VAL_SIZE}
- Test pairs: {TEST_SIZE}
- Synthetic pairs requested: {SYNTHETIC_SIZE}
- Usable synthetic pairs generated: {len(synthetic_pairs)}
- Batch size: {BATCH_SIZE}
- Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}
- Precision: fp32 (`FP16=False`)
- Optimizer: Adafactor
- Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}

## Results

- Baseline BLEU: {baseline_bleu["bleu"]:.4f}
- Improved BLEU: {improved_bleu["bleu"]:.4f}
- Baseline validation loss: {baseline_summary["validation_loss"]:.4f}
- Improved validation loss: {improved_summary["validation_loss"]:.4f}
- Baseline training time: {baseline_summary["train_runtime_seconds"]:.2f} seconds
- Improved training time: {improved_summary["train_runtime_seconds"]:.2f} seconds

## Notes for Report

Back-translation was implemented by training a reverse Bangla-to-English mT5-small model on the original parallel training subset. The reverse model translated Bangla monolingual sentences into synthetic English sources. These synthetic English-Bangla pairs were then appended to the original parallel training data for the improved forward English-to-Bangla model.

Do not interpret the result without checking the sample translations. BLEU can increase even when some individual outputs remain ungrammatical or semantically weak. If Colab memory limits forced smaller settings than planned, report the actual values shown in `training_configuration.csv` and this note.
'''
(RESULTS_DIR / "hardware_and_limitations.md").write_text(discussion, encoding="utf-8")
print(discussion)

In [ ]:
# Package all measured outputs for download.
zip_path = shutil.make_archive(str(RESULTS_DIR), "zip", root_dir=RESULTS_DIR)
print("Created:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print("Download helper unavailable outside Colab:", exc)
    print("Results directory:", RESULTS_DIR)